In [ ]:
from pyspark.sql import SparkSession, functions as F
from IPython.display import display
import time

SPARK_TUNING = {
    "spark.executor.memory":    "6g",
    "spark.executor.cores":     "4",     
    "spark.executor.instances": "2",     
    "spark.dynamicAllocation.enabled": "false",

    "spark.driver.memory":        "4g",
    "spark.driver.maxResultSize": "2g",  

    "spark.sql.shuffle.partitions": "64",
    "spark.default.parallelism":    "64",

    "spark.sql.adaptive.enabled":                    "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.skewJoin.enabled":           "true",

    "spark.sql.autoBroadcastJoinThreshold": str(20 * 1024 * 1024),

    "spark.sql.files.maxPartitionBytes": str(64 * 1024 * 1024),
    "spark.sql.files.openCostInBytes":   str(2 * 1024 * 1024),

    "spark.memory.fraction":        "0.8",
    "spark.memory.storageFraction": "0.2",

    "spark.sql.warehouse.dir": "hdfs:///user/hive/warehouse",
}

builder = SparkSession.builder.appName("olist-05-optimization").enableHiveSupport()
for k, v in SPARK_TUNING.items():
    builder = builder.config(k, v)
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")

BUCKET = "gs://olist-sujinan-2026"
DB     = "olist"

def show(df, n=20):
    display(df.limit(n).toPandas())

def pdf(df, n=1000):
    return df.limit(n).toPandas()

assert spark.sparkContext.appName == "olist-05-optimization", \
    "session เก่าค้างอยู่: เปลี่ยน kernel เป็น Python 3 แล้ว Restart Kernel"

for k in ["spark.executor.instances", "spark.dynamicAllocation.enabled",
          "spark.sql.shuffle.partitions", "spark.sql.warehouse.dir"]:
    print(f"{k:<34s} {spark.conf.get(k)}")

# Recover Silver/Gold from GCS (new cluster only)

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB} "
          f"LOCATION 'hdfs:///user/hive/warehouse/{DB}.db'")

layers = {
    "silver": ["customers", "orders", "order_items", "payments",
               "products", "sellers", "reviews", "geolocation"],
    "gold":   ["full_orders", "customer_spending", "seller_performance",
               "product_metrics", "monthly_trend", "customer_retention",
               "top_products_per_seller"],
}

for layer, names in layers.items():
    for n in names:
        (spark.read.parquet(f"{BUCKET}/{layer}/{n}")
              .write.mode("overwrite").format("parquet")
              .saveAsTable(f"{DB}.{layer}_{n}"))
        print(f"{layer}_{n:<24s} {spark.table(f'{DB}.{layer}_{n}').count():>9,}")

# Timer and Table Loading

In [ ]:
def timed(label, fn, runs=3):
    """รัน fn หลายรอบ คืนเวลาที่น้อยที่สุด และพิมพ์ทุกรอบให้เห็น noise"""
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    print(f"{label:<30s} best {min(times):6.2f}s   runs {[round(t, 2) for t in times]}")
    return min(times)

orders    = spark.table(f"{DB}.silver_orders")
items     = spark.table(f"{DB}.silver_order_items")
customers = spark.table(f"{DB}.silver_customers")
sellers   = spark.table(f"{DB}.silver_sellers")
payments  = spark.table(f"{DB}.silver_payments")
full      = spark.table(f"{DB}.gold_full_orders")

print("executors:", spark.sparkContext._jsc.sc().getExecutorMemoryStatus().size() - 1)

# Comparing 4 types of joins using real data

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# 1) sort-merge: shuffle ทั้งสองฝั่งตาม key > sort > merge
timed("sort-merge (default)", lambda: items.join(sellers, "seller_id").count())

# 2) broadcast: ส่งตารางเล็กไปทุก executor ไม่ shuffle ตารางใหญ่เลย
timed("broadcast hint",       lambda: items.join(F.broadcast(sellers), "seller_id").count())

# 3) shuffle hash: shuffle ทั้งสองฝั่ง แต่สร้าง hash table จากฝั่งเล็กแทนการ sort
timed("shuffle_hash hint",    lambda: items.join(sellers.hint("shuffle_hash"), "seller_id").count())

# 4) merge hint: บังคับ sort-merge แม้ Spark อยากใช้วิธีอื่น
timed("merge hint",           lambda: items.join(sellers.hint("merge"), "seller_id").count())

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(20 * 1024 * 1024))

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print("=== sort-merge ===")
items.join(sellers, "seller_id").explain()
print("\n=== broadcast ===")
items.join(F.broadcast(sellers), "seller_id").explain()
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(20 * 1024 * 1024))

# Expanded to 20 million to clearly demonstrate the difference

In [ ]:
big   = spark.range(20_000_000).withColumn("k", F.col("id") % 3000)
small = spark.range(3000).withColumnRenamed("id", "k").withColumn("label", F.lit("s"))

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

timed("20M sort-merge",   lambda: big.join(small, "k").count())
timed("20M broadcast",    lambda: big.join(F.broadcast(small), "k").count())
timed("20M shuffle_hash", lambda: big.join(small.hint("shuffle_hash"), "k").count())

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(20 * 1024 * 1024))

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print("=== sort-merge ===")
big.join(small, "k").explain()
print("\n=== broadcast ===")
big.join(F.broadcast(small), "k").explain()
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(20 * 1024 * 1024))

# Bucketing: Pay the shuffle cost once during the write phase

In [ ]:
N_BUCKETS = 16  

for name, df in {"orders": orders, "order_items": items}.items():
    (df.repartition(N_BUCKETS, "order_id")     # 1 ไฟล์ต่อ bucket
       .write.mode("overwrite")
       .bucketBy(N_BUCKETS, "order_id")
       .sortBy("order_id")
       .format("parquet")
       .saveAsTable(f"{DB}.bucketed_{name}"))

b_orders = spark.table(f"{DB}.bucketed_orders")
b_items  = spark.table(f"{DB}.bucketed_order_items")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

timed("SMJ ตารางปกติ",      lambda: orders.join(items, "order_id").count())
timed("SMJ ตาราง bucketed", lambda: b_orders.join(b_items, "order_id").count())

print("\n=== ปกติ ===")
orders.join(items, "order_id").explain()
print("\n=== bucketed ===")
b_orders.join(b_items, "order_id").explain()

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(20 * 1024 * 1024))

# Data skew: AQE vs. Salting

In [ ]:
per_seller = full.groupBy("seller_id").count()

show(per_seller.orderBy(F.desc("count")), 10)
show(per_seller.agg(
    F.max("count").alias("max"),
    F.expr("percentile_approx(count, 0.5)").alias("median"),
    F.round(F.avg("count"), 1).alias("avg"),
))

In [ ]:
SALT_N = 8

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

left = full.withColumn("salt", (F.rand(seed=42) * SALT_N).cast("int"))

right = sellers.withColumn("salt", F.explode(F.array([F.lit(i) for i in range(SALT_N)])))

timed("SMJ ปกติ (ให้ AQE จัดการ)", lambda: full.join(sellers, "seller_id").count())
timed("SMJ + salting",            lambda: left.join(right, ["seller_id", "salt"]).count())

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(20 * 1024 * 1024))

# Caching: How effective is it really?

In [ ]:
from pyspark import StorageLevel

def three_queries(df):
    df.groupBy("customer_state").count().collect()
    df.groupBy("seller_id").agg(F.sum("price")).collect()
    df.filter(F.col("review_score") <= 2).count()

before = timed("3 queries ไม่ cache", lambda: three_queries(full))

cached = full.persist(StorageLevel.MEMORY_AND_DISK)
cached.count()
after = timed("3 queries หลัง cache", lambda: three_queries(cached))

print(f"เร็วขึ้น {before / after:.1f} เท่า  ลดเวลา {(before - after) / before:.0%}")

In [ ]:
cached.unpersist()

# Hive table 

In [ ]:
show(spark.sql(f"SHOW TABLES IN {DB}"), 30)

show(spark.sql(f"""
    SELECT customer_segment,
           COUNT(*)                     AS orders,
           ROUND(SUM(order_revenue), 2) AS revenue,
           ROUND(AVG(review_score), 2)  AS avg_review
    FROM {DB}.gold_full_orders
    GROUP BY customer_segment
    ORDER BY revenue DESC
"""))

show(spark.sql(f"DESCRIBE FORMATTED {DB}.gold_full_orders"), 40)

# Partitioned Parquet on GCS

In [ ]:
full = spark.table(f"{DB}.gold_full_orders")

(full
   .withColumn("order_year",  F.year("order_purchase_timestamp"))
   .withColumn("order_month", F.month("order_purchase_timestamp"))
   .repartition("order_year", "order_month")      
   .write.mode("overwrite")
   .partitionBy("order_year", "order_month")
   .parquet(f"{BUCKET}/serving/full_orders"))

q = (spark.read.parquet(f"{BUCKET}/serving/full_orders")
        .filter((F.col("order_year") == 2018) & (F.col("order_month") == 3)))
q.explain()         
print(q.count())

# CSV for Excel Users 

In [ ]:
for name in ["monthly_trend", "seller_performance", "product_metrics"]:
    (spark.table(f"{DB}.gold_{name}")
        .coalesce(1)
        .write.mode("overwrite")
        .option("header", True)
        .csv(f"{BUCKET}/serving/csv/{name}"))